In [1]:
import pandas as pd

# Load subscriptions table
subs = pd.read_csv("../data/ravenstack_subscriptions.csv")

# Quick inspection
print("Shape:", subs.shape)
subs.head()

Shape: (5000, 14)


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,23/12/2023,12/04/2024,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,11/06/2024,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,25/11/2024,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,23/11/2024,13/12/2024,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,10/01/2024,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


In [2]:
print("Shape:", subs.shape)
subs.head()

subs.info()
subs.describe()


Shape: (5000, 14)
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   subscription_id    5000 non-null   str  
 1   account_id         5000 non-null   str  
 2   start_date         5000 non-null   str  
 3   end_date           486 non-null    str  
 4   plan_tier          5000 non-null   str  
 5   seats              5000 non-null   int64
 6   mrr_amount         5000 non-null   int64
 7   arr_amount         5000 non-null   int64
 8   is_trial           5000 non-null   bool 
 9   upgrade_flag       5000 non-null   bool 
 10  downgrade_flag     5000 non-null   bool 
 11  churn_flag         5000 non-null   bool 
 12  billing_frequency  5000 non-null   str  
 13  auto_renew_flag    5000 non-null   bool 
dtypes: bool(5), int64(3), str(6)
memory usage: 376.1 KB


,seats,mrr_amount,arr_amount
count,5000.000000,5000.000000,5000.000000
mean,29.852000,2267.749400,27212.992800
std,23.089771,3421.375348,41056.504178
min,1.000000,0.000000,0.000000
25%,14.000000,285.000000,3420.000000
50%,24.000000,931.000000,11172.000000
75%,40.000000,2786.000000,33432.000000
max,189.000000,33830.000000,405960.000000


In [3]:
subs.isnull().sum()

subscription_id         0
account_id              0
start_date              0
end_date             4514
plan_tier               0
seats                   0
mrr_amount              0
arr_amount              0
is_trial                0
upgrade_flag            0
downgrade_flag          0
churn_flag              0
billing_frequency       0
auto_renew_flag         0
dtype: int64

In [4]:
# Check churned subscriptions
subs[subs["churn_flag"] == 1][["churn_flag", "end_date"]].head()

# Check active subscriptions
subs[subs["churn_flag"] == 0][["churn_flag", "end_date"]].head()

,churn_flag,end_date
1,False,NaN
2,False,NaN
4,False,NaN
5,False,NaN
6,False,NaN


In [5]:
subs[subs["churn_flag"] == True][["churn_flag", "end_date"]].head()

,churn_flag,end_date
0,True,12/04/2024
3,True,13/12/2024
16,True,03/08/2024
17,True,13/08/2024
21,True,27/12/2024


In [6]:
usage = pd.read_csv("../data/ravenstack_feature_usage.csv")

print("Shape:", usage.shape)
usage.head()

Shape: (25000, 8)


,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,27/07/2023,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,07/08/2023,feature_5,9,369,0,False
2,U-096807,S-f29e7f,07/12/2023,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,28/07/2024,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,02/12/2024,feature_12,12,900,0,False


In [7]:
usage.columns

Index(['usage_id', 'subscription_id', 'usage_date', 'feature_name',
       'usage_count', 'usage_duration_secs', 'error_count', 'is_beta_feature'],
      dtype='str')

In [8]:
# Convert usage_date to datetime
usage["usage_date"] = pd.to_datetime(usage["usage_date"], dayfirst=True)

# Aggregate usage per subscription
usage_agg = usage.groupby("subscription_id").agg(
    total_usage_count=("usage_count", "sum"),
    total_usage_duration=("usage_duration_secs", "sum"),
    avg_usage_duration=("usage_duration_secs", "mean"),
    total_errors=("error_count", "sum"),
    beta_feature_usage_rate=("is_beta_feature", "mean"),
    unique_features_used=("feature_name", "nunique")
).reset_index()

usage_agg.head()

,subscription_id,total_usage_count,total_usage_duration,avg_usage_duration,total_errors,beta_feature_usage_rate,unique_features_used
0,S-001561,48,21604,4320.80,3,0.00,5
1,S-0027d3,44,20848,5212.00,0,0.00,3
2,S-003647,71,13018,1627.25,3,0.25,8
3,S-003fc0,43,11933,2983.25,1,0.00,4
4,S-004d19,18,5616,2808.00,0,0.00,2


In [9]:
# Merge behavioral features into subscriptions
subs_merged = subs.merge(usage_agg, on="subscription_id", how="left")

print("Shape after merge:", subs_merged.shape)
subs_merged.head()

Shape after merge: (5000, 20)


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,total_usage_count,total_usage_duration,avg_usage_duration,total_errors,beta_feature_usage_rate,unique_features_used
0,S-8cec59,A-3c1a3f,23/12/2023,12/04/2024,Enterprise,14,2786,33432,False,False,False,True,monthly,True,63.0,26418.0,4403.000000,2.0,0.166667,6.0
1,S-0f6f44,A-9b9fe9,11/06/2024,NaN,Pro,17,833,9996,False,False,False,False,monthly,True,36.0,6819.0,1704.750000,6.0,0.000000,4.0
2,S-51c0d1,A-659280,25/11/2024,NaN,Enterprise,62,0,0,True,True,False,False,annual,False,25.0,13877.0,4625.666667,2.0,0.000000,3.0
3,S-f81687,A-e7a1e2,23/11/2024,13/12/2024,Enterprise,5,995,11940,False,False,False,True,monthly,True,65.0,22943.0,3277.571429,5.0,0.000000,7.0
4,S-cff5a2,A-ba6516,10/01/2024,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
usage_cols = [
    "total_usage_count",
    "total_usage_duration",
    "avg_usage_duration",
    "total_errors",
    "beta_feature_usage_rate",
    "unique_features_used"
]

subs_merged[usage_cols] = subs_merged[usage_cols].fillna(0)

In [11]:
tickets = pd.read_csv("../data/ravenstack_support_tickets.csv")

print("Shape:", tickets.shape)
tickets.head()

Shape: (2000, 9)


,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,27/07/2023,28/07/2023 03:00,27,high,74,NaN,False
1,T-4d04b9,A-e43bf7,08/07/2024,09/07/2024 03:00,27,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,17/10/2024,17/10/2024 19:00,19,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,08/09/2024,09/09/2024 23:00,47,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,30/11/2024,01/12/2024 02:00,26,medium,8,NaN,False


In [12]:
tickets.columns

Index(['ticket_id', 'account_id', 'submitted_at', 'closed_at',
       'resolution_time_hours', 'priority', 'first_response_time_minutes',
       'satisfaction_score', 'escalation_flag'],
      dtype='str')

In [13]:
tickets["submitted_at"] = pd.to_datetime(tickets["submitted_at"], dayfirst=True)
tickets["closed_at"] = pd.to_datetime(tickets["closed_at"], dayfirst=True)

In [14]:
tickets_agg = tickets.groupby("account_id").agg(
    total_tickets=("ticket_id", "count"),
    avg_resolution_time=("resolution_time_hours", "mean"),
    avg_first_response_time=("first_response_time_minutes", "mean"),
    avg_satisfaction_score=("satisfaction_score", "mean"),
    escalation_rate=("escalation_flag", "mean")
).reset_index()

tickets_agg.head()

,account_id,total_tickets,avg_resolution_time,avg_first_response_time,avg_satisfaction_score,escalation_rate
0,A-00bed1,4,31.750000,106.25,4.0,0.0
1,A-00cac8,2,33.000000,120.50,NaN,0.0
2,A-0158bb,1,32.000000,50.00,3.0,0.0
3,A-016043,3,30.333333,78.00,4.0,0.0
4,A-019782,2,10.000000,107.00,3.0,0.0


In [15]:
subs_full = subs_merged.merge(tickets_agg, on="account_id", how="left")

print("Shape after merging tickets:", subs_full.shape)
subs_full.head()

Shape after merging tickets: (5000, 25)


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,...,total_usage_duration,avg_usage_duration,total_errors,beta_feature_usage_rate,unique_features_used,total_tickets,avg_resolution_time,avg_first_response_time,avg_satisfaction_score,escalation_rate
0,S-8cec59,A-3c1a3f,23/12/2023,12/04/2024,Enterprise,14,2786,33432,False,False,...,26418.0,4403.000000,2.0,0.166667,6.0,4.0,33.25,56.50,3.500000,0.0
1,S-0f6f44,A-9b9fe9,11/06/2024,NaN,Pro,17,833,9996,False,False,...,6819.0,1704.750000,6.0,0.000000,4.0,4.0,33.50,156.50,4.000000,0.0
2,S-51c0d1,A-659280,25/11/2024,NaN,Enterprise,62,0,0,True,True,...,13877.0,4625.666667,2.0,0.000000,3.0,4.0,8.00,123.75,4.333333,0.0
3,S-f81687,A-e7a1e2,23/11/2024,13/12/2024,Enterprise,5,995,11940,False,False,...,22943.0,3277.571429,5.0,0.000000,7.0,1.0,45.00,37.00,4.000000,0.0
4,S-cff5a2,A-ba6516,10/01/2024,NaN,Enterprise,27,5373,64476,False,False,...,0.0,0.000000,0.0,0.000000,0.0,6.0,31.50,71.00,4.000000,0.0


In [16]:
ticket_cols = [
    "total_tickets",
    "avg_resolution_time",
    "avg_first_response_time",
    "avg_satisfaction_score",
    "escalation_rate"
]

subs_full[ticket_cols] = subs_full[ticket_cols].fillna(0)

In [17]:
print("Shape:", subs_full.shape)

Shape: (5000, 25)


In [18]:
subs_full["start_date"] = pd.to_datetime(subs_full["start_date"], dayfirst=True)
subs_full["end_date"] = pd.to_datetime(subs_full["end_date"], dayfirst=True)

In [19]:
reference_date = subs_full["end_date"].max()
reference_date

Timestamp('2024-12-31 00:00:00')

In [20]:
subs_full["tenure_days"] = (
    subs_full["end_date"].fillna(reference_date)
    - subs_full["start_date"]
).dt.days

In [21]:
subs_full[["start_date", "end_date", "tenure_days"]].head()

,start_date,end_date,tenure_days
0,2023-12-23,2024-04-12,111
1,2024-06-11,NaT,203
2,2024-11-25,NaT,36
3,2024-11-23,2024-12-13,20
4,2024-01-10,NaT,356


In [22]:
y = subs_full["churn_flag"]

In [23]:
X = subs_full.drop(
    ["subscription_id", "account_id", "start_date", "end_date", "churn_flag"],
    axis=1
)

In [24]:
X.dtypes

plan_tier                      str
seats                        int64
mrr_amount                   int64
arr_amount                   int64
is_trial                      bool
upgrade_flag                  bool
downgrade_flag                bool
billing_frequency              str
auto_renew_flag               bool
total_usage_count          float64
total_usage_duration       float64
avg_usage_duration         float64
total_errors               float64
beta_feature_usage_rate    float64
unique_features_used       float64
total_tickets              float64
avg_resolution_time        float64
avg_first_response_time    float64
avg_satisfaction_score     float64
escalation_rate            float64
tenure_days                  int64
dtype: object

In [25]:
X_encoded = pd.get_dummies(X, drop_first=True)

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [27]:
print("X_encoded Shape:", X_encoded.shape)


X_encoded Shape: (5000, 22)


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Train churn ratio:")
print(y_train.value_counts(normalize=True))

print("Test churn ratio:")
print(y_test.value_counts(normalize=True))

Train shape: (4000, 22)
Test shape: (1000, 22)
Train churn ratio:
churn_flag
False    0.90275
True     0.09725
Name: proportion, dtype: float64
Test churn ratio:
churn_flag
False    0.903
True     0.097
Name: proportion, dtype: float64


In [29]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=2000)

log_model.fit(X_train, y_train)

c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [30]:
y_prob = log_model.predict_proba(X_test)[:, 1]
y_pred = log_model.predict(X_test)

In [31]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob))

Confusion Matrix:
[[903   0]
 [ 97   0]]

Classification Report:
              precision    recall  f1-score   support

       False       0.90      1.00      0.95       903
        True       0.00      0.00      0.00        97

    accuracy                           0.90      1000
   macro avg       0.45      0.50      0.47      1000
weighted avg       0.82      0.90      0.86      1000


ROC-AUC Score:
0.6643490769599617


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [32]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

log_model.fit(X_train, y_train)

c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [33]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [34]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced"
)

log_model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [35]:
y_prob = log_model.predict_proba(X_test_scaled)[:, 1]
y_pred = log_model.predict(X_test_scaled)

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob))

Confusion Matrix:
[[468 435]
 [ 31  66]]

Classification Report:
              precision    recall  f1-score   support

       False       0.94      0.52      0.67       903
        True       0.13      0.68      0.22        97

    accuracy                           0.53      1000
   macro avg       0.53      0.60      0.44      1000
weighted avg       0.86      0.53      0.62      1000


ROC-AUC Score:
0.6442785217659348


In [36]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, accuracy_score

thresholds = [0.5, 0.6, 0.65, 0.7, 0.75]

for t in thresholds:
    y_pred_custom = (y_prob >= t).astype(int)

    print(f"\nThreshold: {t}")
    print("Recall:", recall_score(y_test, y_pred_custom))
    print("Precision:", precision_score(y_test, y_pred_custom))
    print("Accuracy:", accuracy_score(y_test, y_pred_custom))


Threshold: 0.5
Recall: 0.6804123711340206
Precision: 0.1317365269461078
Accuracy: 0.534

Threshold: 0.6
Recall: 0.41237113402061853
Precision: 0.17316017316017315
Accuracy: 0.752

Threshold: 0.65
Recall: 0.15463917525773196
Precision: 0.19736842105263158
Accuracy: 0.857

Threshold: 0.7
Recall: 0.010309278350515464
Precision: 0.08333333333333333
Accuracy: 0.893

Threshold: 0.75
Recall: 0.010309278350515464
Precision: 0.3333333333333333
Accuracy: 0.902


In [37]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf = rf_model.predict(X_test)

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob_rf))

Confusion Matrix:
[[903   0]
 [ 97   0]]

Classification Report:
              precision    recall  f1-score   support

       False       0.90      1.00      0.95       903
        True       0.00      0.00      0.00        97

    accuracy                           0.90      1000
   macro avg       0.45      0.50      0.47      1000
weighted avg       0.82      0.90      0.86      1000


ROC-AUC Score:
0.6767932778481808


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [38]:
import numpy as np

print("Max probability:", np.max(y_prob_rf))
print("Mean probability:", np.mean(y_prob_rf))

Max probability: 0.34
Mean probability: 0.09237000000000001


In [39]:
thresholds = [0.15, 0.2, 0.25, 0.3]

for t in thresholds:
    y_pred_custom = (y_prob_rf >= t).astype(int)

    print(f"\nThreshold: {t}")
    print("Recall:", recall_score(y_test, y_pred_custom))
    print("Precision:", precision_score(y_test, y_pred_custom))
    print("Accuracy:", accuracy_score(y_test, y_pred_custom))


Threshold: 0.15
Recall: 0.3402061855670103
Precision: 0.19411764705882353
Accuracy: 0.799

Threshold: 0.2
Recall: 0.14432989690721648
Precision: 0.21212121212121213
Accuracy: 0.865

Threshold: 0.25
Recall: 0.020618556701030927
Precision: 0.08
Accuracy: 0.882

Threshold: 0.3
Recall: 0.010309278350515464
Precision: 0.14285714285714285
Accuracy: 0.898


In [40]:
# 1️⃣ Revenue Per Seat
# Enterprise logic
subs_full["revenue_per_seat"] = subs_full["mrr_amount"] / subs_full["seats"]

In [41]:
# 2️⃣ Usage Per Seat
subs_full["usage_per_seat"] = subs_full["total_usage_count"] / subs_full["seats"]

In [42]:
#3️⃣ Error Rate
subs_full["error_rate"] = subs_full["total_errors"] / (subs_full["total_usage_count"] + 1)

In [43]:
# 4️⃣ Engagement Intensity Per Day
subs_full["usage_per_day"] = subs_full["total_usage_count"] / (subs_full["tenure_days"] + 1)

In [44]:
y = subs_full["churn_flag"]

X = subs_full.drop(
    ["subscription_id", "account_id", "start_date", "end_date", "churn_flag"],
    axis=1
)

In [45]:
X_encoded = pd.get_dummies(X, drop_first=True)

print("New X shape:", X_encoded.shape)

New X shape: (5000, 26)


In [46]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [47]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf = rf_model.predict(X_test)

In [48]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob_rf))

Confusion Matrix:
[[903   0]
 [ 97   0]]

Classification Report:
              precision    recall  f1-score   support

       False       0.90      1.00      0.95       903
        True       0.00      0.00      0.00        97

    accuracy                           0.90      1000
   macro avg       0.45      0.50      0.47      1000
weighted avg       0.82      0.90      0.86      1000


ROC-AUC Score:
0.6583153520338848


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [49]:
print("Max RF probability:", y_prob_rf.max())
print("Mean RF probability:", y_prob_rf.mean())

Max RF probability: 0.42
Mean RF probability: 0.0929575
